# Fetch Realisierter Stromverbrauch (Netzlast) from SMARD

Uses the undocumented but public SMARD `chart_data` JSON API:
- **Index endpoint**: `https://www.smard.de/app/chart_data/410/DE/index_hour.json`  
  Returns the list of available weekly bucket timestamps (Unix ms).
- **Data endpoint**: `https://www.smard.de/app/chart_data/410/DE/410_DE_hour_{timestamp}.json`  
  Returns `series` — a list of `[timestamp_ms, value_MWh]` pairs for one week.

Filter IDs used here:
| ID  | Category |
|-----|----------|
| 410 | Realisierter Stromverbrauch – Netzlast |

Region: `DE`, Resolution: `hour`

The function fetches all weekly buckets overlapping with `[start_date, end_date]`,
combines them, clips to the exact range and saves a CSV.


In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone

SMARD_BASE = "https://www.smard.de/app/chart_data"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; smard-fetcher/1.0)"}

# Filter IDs
FILTER_NETZLAST = 410          # Realisierter Stromverbrauch – Netzlast


def _get_index(filter_id: int, region: str = "DE", resolution: str = "hour") -> list[int]:
    """Return the list of weekly bucket timestamps (Unix ms) available for the given filter."""
    url = f"{SMARD_BASE}/{filter_id}/{region}/index_{resolution}.json"
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()["timestamps"]


def _fetch_week(filter_id: int, timestamp_ms: int, region: str = "DE", resolution: str = "hour") -> list:
    """Fetch the raw series [[ts_ms, value], ...] for one weekly bucket."""
    url = f"{SMARD_BASE}/{filter_id}/{region}/{filter_id}_{region}_{resolution}_{timestamp_ms}.json"
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json().get("series", [])


def fetch_smard_netzlast(
    start_date: str,
    end_date: str,
    output_file: str | None = None,
    region: str = "DE",
    resolution: str = "hour",
    filter_id: int = FILTER_NETZLAST,
    sleep: float = 0.3,
) -> pd.DataFrame:
    """
    Fetch Realisierter Stromverbrauch (Netzlast) from the SMARD chart_data API.

    Parameters
    ----------
    start_date : str
        Inclusive start in 'YYYY-MM-DD' format (local CET/CEST time).
    end_date : str
        Inclusive end in 'YYYY-MM-DD' format.
    output_file : str | None
        If given, save the result as CSV to this path.
    region : str
        SMARD region code, default 'DE'.
    resolution : str
        'hour' or 'quarterhour'.
    filter_id : int
        SMARD filter ID (410 = Netzlast).
    sleep : float
        Seconds to sleep between requests (be polite to the server).

    Returns
    -------
    pd.DataFrame with columns ['timestamp', 'load_MWh'].
    """
    # Convert date strings to UTC millisecond boundaries
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt   = datetime.strptime(end_date,   "%Y-%m-%d").replace(tzinfo=timezone.utc)
    # Include the full end day
    end_ms   = int(end_dt.timestamp() * 1000) + 86_400_000 - 1
    start_ms = int(start_dt.timestamp() * 1000)

    print(f"Fetching index for filter {filter_id} / {region} / {resolution} ...")
    all_timestamps = _get_index(filter_id, region, resolution)

    # Keep only buckets that can overlap with [start_ms, end_ms].
    # Each bucket covers roughly one week (604_800_000 ms).
    week_ms = 7 * 24 * 3600 * 1000
    relevant = [ts for ts in all_timestamps if ts <= end_ms and ts + week_ms >= start_ms]

    if not relevant:
        print("No data available for the requested period.")
        return pd.DataFrame(columns=["timestamp", "load_MWh"])

    print(f"Fetching {len(relevant)} weekly bucket(s) ...")
    rows = []
    for ts in relevant:
        series = _fetch_week(filter_id, ts, region, resolution)
        rows.extend(series)
        time.sleep(sleep)

    # Build DataFrame and clip to the exact requested range
    df = pd.DataFrame(rows, columns=["ts_ms", "load_MWh"])
    df = df.dropna(subset=["load_MWh"])
    df = df[(df["ts_ms"] >= start_ms) & (df["ts_ms"] <= end_ms)]
    df["timestamp"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True).dt.tz_convert("Europe/Berlin")
    df = df[["timestamp", "load_MWh"]].sort_values("timestamp").reset_index(drop=True)

    print(f"Retrieved {len(df)} rows from {df['timestamp'].iloc[0]} to {df['timestamp'].iloc[-1]}")

    if output_file:
        os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
        df.to_csv(output_file, index=False)
        print(f"Saved to {output_file}")

    return df


In [ ]:
# Example: fetch two weeks of data and save as CSV
df = fetch_smard_netzlast(
    start_date="2026-04-25",
    end_date="2026-05-09",
    output_file="../data/raw/netzlast_2026-04-25_2026-05-09.csv",
)
df.head(10)


In [ ]:
# fetch netzlast since 2025-10-01 in 14 days cycle (search time frame limit on SMARD API)
import time

start_date = datetime(2025, 10, 1)
end_date = datetime.now()
date_ranges = []
while start_date < end_date:
    range_end = min(start_date + pd.Timedelta(days=14), end_date)
    date_ranges.append((start_date.strftime("%Y-%m-%d"), range_end.strftime("%Y-%m-%d")))
    start_date = range_end + pd.Timedelta(days=1)       
for start, end in date_ranges:
    print(f"Fetching data from {start} to {end} ...")
    output_file = f"../data/raw/netzlast_{start}_{end}.csv"
    if os.path.exists(output_file):
        print(f"File {output_file} already exists, skipping.")
        continue
    df = fetch_smard_netzlast(start, end, output_file=output_file) 
    #print(f"Fetched {len(df)} rows for {start} to {end}. saved to {output_file}")
    time.sleep(1)  # be polite to the server

# Combine all fetched CSVs into one DataFrame and save as a single CSV
all_files = [f"../data/raw/netzlast_{start}_{end}.csv" for start, end in date_ranges]
combined_df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
now = datetime.now().strftime("%Y-%m-%d_%H")
print(f"Combined {len(all_files)} files with total {len(combined_df)} rows (as of {now}).") 
combined_df.to_csv(f"../data/raw/netzlast_2025-10-01_to_{now}.csv", index=False)
print(f"Combined data saved to ../data/raw/netzlast_2025-10-01_to_{now}.csv") 

# delete individual files to save space
for start, end in date_ranges:  
    file_path = f"../data/raw/netzlast_{start}_{end}.csv"
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleted intermediate file {file_path}") 
